# Feature Engineering — Transformation des variables ResStock
Conversion des valeurs discrètes/textuelles en valeurs numériques continues.
Entrée : `metadata_clean.parquet` → Sortie : `metadata_features.parquet`

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'

df = pd.read_parquet(DATA_PROCESSED / 'metadata_clean.parquet')
print('Shape original :', df.shape)
df_feat = df.copy()

## 1. Isolation — R-values : `"R-11"` → `11.0`

In [ ]:
def extract_r(val):
    if pd.isna(val) or str(val).strip() in ('None', 'nan'): return np.nan
    s = str(val)
    if 'Uninsulated' in s: return 0.0
    m = re.search(r'R-?(\d+\.?\d*)', s)
    return float(m.group(1)) if m else np.nan

INSUL_COLS = [
    'in.insulation_wall', 'in.insulation_ceiling', 'in.insulation_roof',
    'in.insulation_floor', 'in.insulation_foundation_wall',
    'in.insulation_rim_joist', 'in.insulation_slab',
]
for col in INSUL_COLS:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].apply(extract_r)

print('Exemple insulation_wall :', df_feat['in.insulation_wall'].value_counts().head())

## 2. Surface habitable — midpoint ft² → m² : `"1000-1499"` → `116.1 m²`

In [ ]:
def midpoint_m2(val):
    if pd.isna(val): return np.nan
    s = str(val).replace(',', '')
    if '+' in s:
        return float(s.replace('+', '')) * 0.0929
    m = re.match(r'(\d+)-(\d+)', s)
    if m:
        midpoint = (float(m.group(1)) + float(m.group(2))) / 2
        return round(midpoint * 0.0929, 1)
    return np.nan

if 'in.geometry_floor_area' in df_feat.columns:
    df_feat['in.geometry_floor_area'] = df_feat['in.geometry_floor_area'].apply(midpoint_m2)

print('Surface (m2) — stats :')
print(df_feat['in.geometry_floor_area'].describe().round(1))

## 3. Consignes de température — °F → °C : `"68F"` → `20.0°C`

In [ ]:
def f_to_c(val):
    if pd.isna(val): return np.nan
    m = re.search(r'(\d+\.?\d*)', str(val))
    return round((float(m.group(1)) - 32) * 5 / 9, 1) if m else np.nan

SP_COLS = [
    'in.heating_setpoint', 'in.cooling_setpoint',
    'in.heating_setpoint_offset_magnitude', 'in.cooling_setpoint_offset_magnitude',
]
for col in SP_COLS:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].apply(f_to_c)

print('Consigne chauffage (degC) :', df_feat['in.heating_setpoint'].value_counts().head())
print('Consigne clim (degC)      :', df_feat['in.cooling_setpoint'].value_counts().head())

## 4. Efficacité HVAC — `"AFUE, 80%"` → `0.80` | `"SEER, 13"` → `13.0`

In [ ]:
def extract_efficiency(val):
    if pd.isna(val): return np.nan
    s = str(val)
    m_pct = re.search(r'(\d+\.?\d*)%', s)
    if m_pct: return round(float(m_pct.group(1)) / 100, 3)
    m_num = re.search(r'[,\s]+(\d+\.?\d*)\s*$', s)
    if m_num: return float(m_num.group(1))
    return np.nan

EFF_COLS = ['in.hvac_heating_efficiency', 'in.hvac_cooling_efficiency']
for col in EFF_COLS:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].apply(extract_efficiency)

print('Efficacite chauffage :', df_feat['in.hvac_heating_efficiency'].describe().round(3))

## 5. Variables ordinales simples — string → int

In [ ]:
NUM_COLS = [
    'in.geometry_stories',
    'in.bedrooms',
    'in.occupants',
    'in.air_leakage_to_outside_ach50',
]
for col in NUM_COLS:
    if col in df_feat.columns:
        df_feat[col] = pd.to_numeric(df_feat[col], errors='coerce')

print('Stories   :', df_feat['in.geometry_stories'].value_counts().to_dict())
print('Occupants :', df_feat['in.occupants'].value_counts().sort_index().to_dict())

## 6. Revenu — midpoint : `"10000-14999"` → `12500`

In [ ]:
def midpoint_income(val):
    if pd.isna(val): return np.nan
    s = str(val).replace(',', '').replace('$', '').strip()
    if s.startswith('<'):
        return float(s[1:]) / 2
    if s.endswith('+'):
        return float(s[:-1])
    m = re.match(r'(\d+)-(\d+)', s)
    if m:
        return (float(m.group(1)) + float(m.group(2))) / 2
    return pd.to_numeric(s, errors='coerce')

if 'in.income' in df_feat.columns:
    df_feat['in.income'] = df_feat['in.income'].apply(midpoint_income)

print('Revenu (USD) — stats :')
print(df_feat['in.income'].describe().round(0))

## 7. Résumé — colonnes transformées

In [ ]:
transformed = INSUL_COLS + ['in.geometry_floor_area'] + SP_COLS + EFF_COLS + NUM_COLS + ['in.income']
transformed = [c for c in transformed if c in df_feat.columns]

print(f'{"Colonne":<45} {"Type original":>15} {"Type final":>12} {"% NaN final":>12}')
print('-' * 90)
for col in transformed:
    t_orig = str(df[col].dtype)
    t_new  = str(df_feat[col].dtype)
    pct_nan = df_feat[col].isna().mean() * 100
    print(f'{col:<45} {t_orig:>15} {t_new:>12} {pct_nan:>11.1f}%')

## 8. Sauvegarde → `metadata_features.parquet`

In [ ]:
out_path = DATA_PROCESSED / 'metadata_features.parquet'
df_feat.to_parquet(out_path, index=False)
print(f'Sauvegarde : {out_path}')
print(f'Shape final : {df_feat.shape}')